In [1]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import joblib
import warnings
warnings.filterwarnings('ignore')

In [3]:
SCRIPT_DIR = Path.cwd()
DATA_DIR = SCRIPT_DIR.parent / "data" / "election_data"
MODEL_SAVE_PATH = SCRIPT_DIR / "phase1_base_model.pth"
SCALER_SAVE_PATH = SCRIPT_DIR / "feature_scaler.joblib"
FEATURES_SAVE_PATH = SCRIPT_DIR / "feature_columns.joblib"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
print("Data directory:", DATA_DIR)

Using device: cuda
Data directory: c:\Users\kscho\OneDrive\Documents\Political-Intelligence-System\data\election_data


In [4]:
census = pd.read_csv(DATA_DIR / "census_2024_435_cleaned.csv")
history = pd.read_csv(DATA_DIR / "district_history_cleaned.csv")
pvi = pd.read_csv(DATA_DIR / "state_pvi_cleaned.csv")
house = pd.read_csv(DATA_DIR / "house_results_cleaned.csv", low_memory=False)

print(census.shape)
print(history.shape)
print(pvi.shape)
print(house.shape)

(435, 246)
(468, 23)
(50, 6)
(18388, 22)


In [5]:
house['winner'] = (house['party'] == 'R').astype(int)
print(house['winner'].value_counts())

winner
0    18388
Name: count, dtype: int64


In [6]:
house = house.merge(census, on='district_id', how='left')
house = house.merge(pvi, left_on='state_x', right_on='state', how='left')
house = house.merge(history, on='district_id', how='left')

house['state'] = house['state_x']
house = house.drop(columns=['state_x', 'state_y'], errors='ignore')

print(f"Shape after merges: {house.shape}")

Shape after merges: (18477, 294)


In [7]:
categorical_cols = house.select_dtypes(include=['object', 'str']).columns.tolist()
categorical_cols = [col for col in categorical_cols if col not in ['candidate_name', 'district_id']]

for col in categorical_cols:
    if col in house.columns:
        le = LabelEncoder()
        house[col + '_encoded'] = le.fit_transform(house[col].astype(str))
        print(f"Encoded: {col}")

Encoded: state_abbr_x
Encoded: office
Encoded: stage
Encoded: runoff
Encoded: party
Encoded: mode
Encoded: party_clean
Encoded: district_name
Encoded: NAME
Encoded: state_abbr_y
Encoded: state
Encoded: pvi_label
Encoded: state_lean
Encoded: winner_2004
Encoded: winner_2006
Encoded: winner_2008
Encoded: winner_2010
Encoded: winner_2012
Encoded: winner_2014
Encoded: winner_2016
Encoded: winner_2018
Encoded: winner_2020
Encoded: winner_2022
Encoded: winner_2024


In [8]:
exclude_cols = ['winner', 'candidate_name', 'district_id', 'notes', 'state', 'party']
feature_cols = []

for col in house.columns:
    if col not in exclude_cols and 'winner' not in col:
        if col.endswith('_encoded') or pd.api.types.is_numeric_dtype(house[col]):
            feature_cols.append(col)

X = house[feature_cols].copy()
y = house['winner'].copy()

for col in X.columns:
    X[col] = pd.to_numeric(X[col], errors='coerce')

X = X.fillna(0)
X = X.replace([np.inf, -np.inf], 0)

print(f"Features shape: {X.shape}")
print(f"Number of features: {len(feature_cols)}")

Features shape: (18477, 280)
Number of features: 280


In [9]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training samples: {len(X_train)}")
print(f"Validation samples: {len(X_val)}")

Training samples: 14781
Validation samples: 3696


In [10]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

joblib.dump(scaler, SCALER_SAVE_PATH)
joblib.dump(feature_cols, FEATURES_SAVE_PATH)
print(f"Scaler saved to {SCALER_SAVE_PATH}")
print(f"Feature columns saved to {FEATURES_SAVE_PATH}")

Scaler saved to c:\Users\kscho\OneDrive\Documents\Political-Intelligence-System\models\feature_scaler.joblib
Feature columns saved to c:\Users\kscho\OneDrive\Documents\Political-Intelligence-System\models\feature_columns.joblib


In [11]:
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).reshape(-1, 1)
X_val_tensor = torch.tensor(X_val_scaled, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val.values, dtype=torch.float32).reshape(-1, 1)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
val_dataset = TensorDataset(X_val_tensor, y_val_tensor)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

In [12]:
class ElectionPredictor(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        return self.network(x)

input_dim = X_train_scaled.shape[1]
model = ElectionPredictor(input_dim).to(device)
print(f"Input dimension: {input_dim}")

Input dimension: 280


In [13]:
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=5, factor=0.5)

num_epochs = 25
best_val_loss = float('inf')
patience_counter = 0
early_stop_patience = 15

print("\nStarting training...")
print("=" * 60)

for epoch in range(num_epochs):
    model.train()
    train_loss = 0
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
    
    model.eval()
    val_loss = 0
    val_preds = []
    val_targets = []
    
    with torch.no_grad():
        for batch_X, batch_y in val_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            val_loss += loss.item()
            
            val_preds.extend((outputs.cpu().numpy() > 0.5).astype(int).flatten())
            val_targets.extend(batch_y.cpu().numpy().flatten())
    
    train_loss /= len(train_loader)
    val_loss /= len(val_loader)
    val_accuracy = accuracy_score(val_targets, val_preds)
    
    scheduler.step(val_loss)
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1}/{num_epochs}]")
        print(f"  Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")
        print(f"  Val Accuracy: {val_accuracy:.4f}")
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), MODEL_SAVE_PATH)
        patience_counter = 0
        print(f"  Model saved with val_loss: {val_loss:.4f}")
    else:
        patience_counter += 1
    
    if patience_counter >= early_stop_patience:
        print(f"\nEarly stopping at epoch {epoch+1}")
        break

print("\n" + "=" * 60)
print("Training Complete!")


Starting training...
  Model saved with val_loss: 0.0475
  Model saved with val_loss: 0.0108
  Model saved with val_loss: 0.0045
  Model saved with val_loss: 0.0025
  Model saved with val_loss: 0.0016
  Model saved with val_loss: 0.0010
  Model saved with val_loss: 0.0007
  Model saved with val_loss: 0.0005
  Model saved with val_loss: 0.0004
Epoch [10/25]
  Train Loss: 0.0005, Val Loss: 0.0003
  Val Accuracy: 1.0000
  Model saved with val_loss: 0.0003
  Model saved with val_loss: 0.0003
  Model saved with val_loss: 0.0002
  Model saved with val_loss: 0.0002
  Model saved with val_loss: 0.0001
  Model saved with val_loss: 0.0001
  Model saved with val_loss: 0.0001
  Model saved with val_loss: 0.0001
  Model saved with val_loss: 0.0001
  Model saved with val_loss: 0.0001
Epoch [20/25]
  Train Loss: 0.0001, Val Loss: 0.0001
  Val Accuracy: 1.0000
  Model saved with val_loss: 0.0001
  Model saved with val_loss: 0.0000
  Model saved with val_loss: 0.0000
  Model saved with val_loss: 0.000

In [15]:
model.load_state_dict(torch.load(MODEL_SAVE_PATH))
model.eval()

with torch.no_grad():
    val_preds = []
    val_targets = []
    for batch_X, batch_y in val_loader:
        batch_X = batch_X.to(device)
        outputs = model(batch_X)
        val_preds.extend((outputs.cpu().numpy() > 0.5).astype(int).flatten())
        val_targets.extend(batch_y.cpu().numpy().flatten())

val_acc = accuracy_score(val_targets, val_preds)
val_precision = precision_score(val_targets, val_preds, zero_division=0)
val_recall = recall_score(val_targets, val_preds, zero_division=0)
val_f1 = f1_score(val_targets, val_preds, zero_division=0)

print("\nFinal Model Performance:")
print("-" * 40)
print(f"Validation Accuracy: {val_acc:.4f}")
print(f"Validation Precision: {val_precision:.4f}")
print(f"Validation Recall: {val_recall:.4f}")
print(f"Validation F1-Score: {val_f1:.4f}")

print("\nConfusion Matrix (Validation):")
cm = confusion_matrix(val_targets, val_preds)
print(cm)

if cm.shape == (2, 2):
    print(f"\nTrue Negatives (Democrat): {cm[0,0]}, False Positives: {cm[0,1]}")
    print(f"False Negatives: {cm[1,0]}, True Positives (Republican): {cm[1,1]}")
else:
    print(f"\nOnly one class predicted. Matrix shape: {cm.shape}")
    if len(np.unique(val_preds)) == 1:
        print(f"Model predicted only class: {val_preds[0]}")
        print(f"Actual class distribution: Democrat={sum(np.array(val_targets)==0)}, Republican={sum(np.array(val_targets)==1)}")

print(f"\nModel saved to: {MODEL_SAVE_PATH}")
print(f"Scaler saved to: {SCALER_SAVE_PATH}")
print(f"Feature columns saved to: {FEATURES_SAVE_PATH}")


Final Model Performance:
----------------------------------------
Validation Accuracy: 1.0000
Validation Precision: 0.0000
Validation Recall: 0.0000
Validation F1-Score: 0.0000

Confusion Matrix (Validation):
[[3696]]

Only one class predicted. Matrix shape: (1, 1)
Model predicted only class: 0
Actual class distribution: Democrat=3696, Republican=0

Model saved to: c:\Users\kscho\OneDrive\Documents\Political-Intelligence-System\models\phase1_base_model.pth
Scaler saved to: c:\Users\kscho\OneDrive\Documents\Political-Intelligence-System\models\feature_scaler.joblib
Feature columns saved to: c:\Users\kscho\OneDrive\Documents\Political-Intelligence-System\models\feature_columns.joblib
